# Duka Electronics: Half-Year Sales Exploration

**Analyst:** Timothy Kerre | **Cohort:** 1milliondevs 2026 | **Assignment:** PLP Data Analytics A3, Engagement 1

**Purpose:** The Head of Sales is preparing the January to June 2026 review and wants "answers, not just tables". This notebook explores 1,000 Duka Electronics orders across five branches and answers her questions, each with the code and a written answer beneath it.

## Executive Summary

- Duka Electronics earned KES 71,792,500 from 1,000 orders between 1 January and 29 June 2026, an average of KES 71,792.50 per order.
- Nairobi is the strongest branch, with KES 26,106,100 in revenue (36.4% of the total, almost double Mombasa's 19.5%) and the most orders (376 of 1,000).
- Laptops generate 57.9% of revenue (KES 41,535,000) from only 22.0% of orders, while Tablets sell the most units (691 against 639 for Laptops).
- Premium online orders from Nairobi and Mombasa total 83 orders worth KES 10,218,000 (14.2% of all revenue), and online orders are worth as much as in-store ones (KES 71,207 against KES 70,759 on average), so growing online sales is worthwhile.
- Revenue peaked in April at KES 13,518,300 and was lowest in June at KES 10,903,400, a 24.0% gap, so promotions should target the May to June slowdown.

## Setup and data loading

**Approach:** All imports are in one cell. The raw CSV is not committed to the repository, so if it is missing, `generate_data.py` recreates it (seed 2026, unchanged). In Colab, the repository is cloned first so the helper files and the Excel workbook are available.

In [1]:
import os
import subprocess
import sys

# In Colab only the notebook is opened, so fetch the repository for the other files.
if not os.path.exists("generate_data.py"):
    subprocess.run(["git", "clone", "https://github.com/kerre-star/plp-da-a3-duka-sales-and-agent-cleanup.git"], check=True)
    os.chdir("plp-da-a3-duka-sales-and-agent-cleanup")

# Regenerate the raw data if it is not already present.
if not os.path.exists("duka_sales.csv"):
    subprocess.run([sys.executable, "generate_data.py"], check=True)

import numpy as np
import pandas as pd

sales = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
print("Loaded duka_sales.csv:", sales.shape)

Loaded duka_sales.csv: (1000, 7)


## Task 1: Load the data and the first-five-minutes inspection

**Approach:** The file was loaded with `parse_dates=["order_date"]` so dates are real datetimes from the start. Before any analysis, the data is inspected with `.head()`, `.tail()`, `.shape`, `.info()`, `.describe()`, `.nunique()`, `value_counts(normalize=True)` and the date range. Each result is interpreted in the Markdown cell below the code.

In [2]:
print("shape:", sales.shape, "\n")
print("head:")
display(sales.head())
print("tail:")
display(sales.tail())
print("info:")
sales.info()
print("\ndescribe:")
display(sales.describe())
print("distinct values per column:")
print(sales.nunique())
print("\nMissing values in total:", sales.isna().sum().sum())

shape: (1000, 7) 

head:


,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


tail:


,order_id,order_date,branch,product,quantity,channel,unit_price
995,1996,2026-03-04,Mombasa,Laptop,5,In-store,65000
996,1997,2026-03-06,Nairobi,Tablet,2,Phone order,28000
997,1998,2026-01-20,Nairobi,Laptop,3,Online,65000
998,1999,2026-03-02,Nairobi,Headphones,2,In-store,2500
999,2000,2026-01-30,Nakuru,Charger,4,In-store,1200


info:
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[us]
 2   branch      1000 non-null   str           
 3   product     1000 non-null   str           
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   str           
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[us](1), int64(3), str(3)
memory usage: 54.8 KB

describe:


,order_id,order_date,quantity,unit_price
count,1000.000000,1000,1000.000000,1000.000000
mean,1500.500000,2026-03-30 08:44:09.600000,3.012000,23918.300000
min,1001.000000,2026-01-01 00:00:00,1.000000,1200.000000
25%,1250.750000,2026-02-14 00:00:00,2.000000,2500.000000
50%,1500.500000,2026-03-29 00:00:00,3.000000,15000.000000
75%,1750.250000,2026-05-14 00:00:00,4.000000,28000.000000
max,2000.000000,2026-06-29 00:00:00,5.000000,65000.000000
std,288.819436,NaN,1.416991,23908.790546


distinct values per column:
order_id      1000
order_date     179
branch           5
product          5
quantity         5
channel          3
unit_price       5
dtype: int64

Missing values in total: 0


In [3]:
print("Branch mix (share of orders):")
print((sales["branch"].value_counts(normalize=True) * 100).round(1).astype(str) + "%", "\n")
print("Channel mix (share of orders):")
print((sales["channel"].value_counts(normalize=True) * 100).round(1).astype(str) + "%", "\n")
print("Earliest order:", sales["order_date"].min().date())
print("Latest order  :", sales["order_date"].max().date())
print("Days covered  :", (sales["order_date"].max() - sales["order_date"].min()).days + 1)

Branch mix (share of orders):
branch
Nairobi    37.6%
Mombasa    18.6%
Nakuru     16.2%
Kisumu     15.9%
Eldoret    11.7%
Name: proportion, dtype: str 

Channel mix (share of orders):
channel
In-store       57.9%
Online         32.0%
Phone order    10.1%
Name: proportion, dtype: str 

Earliest order: 2026-01-01
Latest order  : 2026-06-29
Days covered  : 180


**Inspection notes:**

- **`.shape`:** 1,000 rows and 7 columns, one row per order, so each order appears once.
- **`.head()` and `.tail()`:** order IDs run from 1001 to 2000, but the dates jump around (3 June, then 2 February, then 5 January), so the file is **not sorted by date**. Any time-based analysis must sort or group by date explicitly.
- **`.info()`:** all 7 columns have 1,000 non-null values, so there are **no missing values**. `order_date` is `datetime64` because of `parse_dates`, and `branch`, `product` and `channel` are text.
- **`.describe()`:** quantity runs from 1 to 5 units (mean 3). Unit price runs from KES 1,200 to KES 65,000, and the mean of 23,918 sits well below the median of 15,000, so a few expensive products pull the average up. There are no zero or negative values.
- **`.nunique()`:** 5 branches, 5 products, 3 channels and only 5 distinct unit prices, so each product has a single price. There are 179 distinct order dates.
- **Business mix:** Nairobi takes 37.6% of orders and Eldoret 11.7%, and about 58% of orders are in-store, 32% online and 10% by phone, so Duka is a Nairobi-led retailer that sells mainly in the store.
- **Period covered:** 1 January to 29 June 2026 (180 days). June therefore has only 29 days of data, which matters when months are compared in Task 6.

## Task 2: Derived columns

**Approach:** Five columns are added with vectorised operations: `revenue` is quantity multiplied by unit price, `month` and `weekday` come from the `.dt` accessor, `is_weekend` is true for Saturday and Sunday (`dayofweek >= 5`), and `price_band` uses `np.where` with the threshold of KES 28,000 (Premium at or above it, Value otherwise). Each new column is checked with `value_counts()`.

In [4]:
sales["revenue"] = sales["quantity"] * sales["unit_price"]
sales["month"] = sales["order_date"].dt.month_name()
sales["weekday"] = sales["order_date"].dt.day_name()
sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5
sales["price_band"] = np.where(sales["unit_price"] >= 28_000, "Premium", "Value")

month_order = ["January", "February", "March", "April", "May", "June"]
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

print("month:\n", sales["month"].value_counts().reindex(month_order), "\n")
print("weekday:\n", sales["weekday"].value_counts().reindex(weekday_order), "\n")
print("is_weekend:\n", sales["is_weekend"].value_counts(), "\n")
print("price_band:\n", sales["price_band"].value_counts(), "\n")
print("Price by band:", sales.groupby("price_band")["unit_price"].unique().to_dict())
sales[["order_id", "order_date", "month", "weekday", "is_weekend", "unit_price", "price_band", "revenue"]].head()

month:
 month
January     173
February    159
March       186
April       167
May         153
June        162
Name: count, dtype: int64 

weekday:
 weekday
Monday       140
Tuesday      135
Wednesday    137
Thursday     145
Friday       166
Saturday     132
Sunday       145
Name: count, dtype: int64 

is_weekend:
 is_weekend
False    723
True     277
Name: count, dtype: int64 

price_band:
 price_band
Value      565
Premium    435
Name: count, dtype: int64 

Price by band: {'Premium': array([65000, 28000]), 'Value': array([15000,  2500,  1200])}


,order_id,order_date,month,weekday,is_weekend,unit_price,price_band,revenue
0,1001,2026-06-03,June,Wednesday,False,15000,Value,45000
1,1002,2026-02-02,February,Monday,False,65000,Premium,260000
2,1003,2026-01-05,January,Monday,False,28000,Premium,140000
3,1004,2026-04-26,April,Sunday,True,2500,Value,12500
4,1005,2026-03-07,March,Saturday,True,2500,Value,7500


**Checks:** All 1,000 orders fall in the six months January to June, and every weekday has orders. `is_weekend` marks 277 orders as weekend and 723 as weekday. `price_band` splits cleanly by product: Tablet (KES 28,000) and Laptop (KES 65,000) are Premium, which puts the Tablet at exactly the threshold and confirms that "at or above" is applied correctly. In total, 435 orders are Premium and 565 are Value.

## Task 3: Branch ranking by revenue and by number of orders

**Approach:** Revenue and orders are aggregated per branch with named aggregation and ranked both ways. Average order value is added because it links the two measures (revenue = orders x average order value).

In [5]:
branch = (
    sales.groupby("branch")
    .agg(total_revenue=("revenue", "sum"), orders=("order_id", "count"))
)
branch["avg_order_value"] = (branch["total_revenue"] / branch["orders"]).round(0)
branch["revenue_share_%"] = (branch["total_revenue"] / branch["total_revenue"].sum() * 100).round(1)
branch["orders_share_%"] = (branch["orders"] / branch["orders"].sum() * 100).round(1)
branch["revenue_rank"] = branch["total_revenue"].rank(ascending=False).astype(int)
branch["orders_rank"] = branch["orders"].rank(ascending=False).astype(int)
branch = branch.sort_values("total_revenue", ascending=False)

print("Average order value, all orders: KES", f"{sales['revenue'].mean():,.2f}")
print("Rankings identical:", (branch["revenue_rank"] == branch["orders_rank"]).all())
branch

Average order value, all orders: KES 71,792.50
Rankings identical: True


,total_revenue,orders,avg_order_value,revenue_share_%,orders_share_%,revenue_rank,orders_rank
branch,,,,,,,
Nairobi,26106100,376,69431.0,36.4,37.6,1,1
Mombasa,14003700,186,75289.0,19.5,18.6,2,2
Nakuru,11970400,162,73891.0,16.7,16.2,3,3
Kisumu,10868100,159,68353.0,15.1,15.9,4,4
Eldoret,8844200,117,75591.0,12.3,11.7,5,5


**Answer:** **Yes, the two rankings are identical:** Nairobi, Mombasa, Nakuru, Kisumu, then Eldoret.

**Why:** Revenue is orders multiplied by average order value, and the average order values are in a narrow band (KES 68,353 to KES 75,591) while the order counts vary widely (117 to 376). Order volume therefore decides the ranking. Nairobi's 376 orders are more than double Mombasa's 186, and no difference in basket size could change that.

Average order value still matters at the margins:

- **Nairobi** has a below-average order value (KES 69,431 against KES 71,793 overall), so its revenue share (36.4%) is slightly below its order share (37.6%).
- **Mombasa and Eldoret** have the highest values (KES 75,289 and KES 75,591), so their revenue shares (19.5% and 12.3%) are above their order shares (18.6% and 11.7%).
- **Nakuru and Kisumu** have almost the same number of orders (162 and 159), so average order value is what separates them: Nakuru earns 10.1% more revenue (KES 11,970,400 against KES 10,868,100) because its average order is KES 73,891 against KES 68,353. Kisumu's small baskets are its improvement opportunity.

## Task 4: Top three products by revenue and by units sold

**Approach:** Revenue and units (the sum of `quantity`) are aggregated per product, and `nlargest(3)` picks the top three on each measure.

In [6]:
product = (
    sales.groupby("product")
    .agg(total_revenue=("revenue", "sum"), units_sold=("quantity", "sum"), orders=("order_id", "count"))
)
product["revenue_share_%"] = (product["total_revenue"] / product["total_revenue"].sum() * 100).round(1)
product["unit_price"] = sales.groupby("product")["unit_price"].first()

print("Top 3 by revenue:")
display(product.nlargest(3, "total_revenue")[["total_revenue", "revenue_share_%", "units_sold"]])
print("Top 3 by units sold:")
display(product.nlargest(3, "units_sold")[["units_sold", "total_revenue", "revenue_share_%"]])
print("All products:")
product.sort_values("total_revenue", ascending=False)

Top 3 by revenue:


,total_revenue,revenue_share_%,units_sold
product,,,
Laptop,41535000,57.9,639
Tablet,19348000,26.9,691
Phone,8880000,12.4,592


Top 3 by units sold:


,units_sold,total_revenue,revenue_share_%
product,,,
Tablet,691,19348000,26.9
Laptop,639,41535000,57.9
Phone,592,8880000,12.4


All products:


,total_revenue,units_sold,orders,revenue_share_%,unit_price
product,,,,,
Laptop,41535000,639,220,57.9,65000
Tablet,19348000,691,215,26.9,28000
Phone,8880000,592,194,12.4,15000
Headphones,1387500,555,187,1.9,2500
Charger,642000,535,184,0.9,1200


**Answer:** The same three products appear in both lists, but in a different order.

- **By revenue:** Laptop (KES 41,535,000, 57.9%), Tablet (KES 19,348,000, 26.9%) and Phone (KES 8,880,000, 12.4%).
- **By units sold:** Tablet (691), Laptop (639) and Phone (592).

**Why the lists differ:** Laptop and Tablet swap places because of price. Tablets sell 52 more units than Laptops, but a Laptop costs KES 65,000 against KES 28,000 for a Tablet (2.3 times as much), so Laptops earn 2.1 times the Tablets' revenue. Units sold are spread almost evenly (535 to 691 across all five products), so revenue is shaped by price rather than by demand.

The cheap products show this most clearly. Headphones (555 units) and Chargers (535 units) sell nearly as many units as Phones (592) but together bring in only 2.8% of revenue. The business depends on Laptops: they are 22.0% of orders but 57.9% of revenue.

## Task 5: Weekend versus weekday orders

**Approach:** Orders are grouped by `is_weekend` to get the number of orders, total revenue and average order revenue. Because weekends are only two days of every seven, the totals are not directly comparable, so orders and revenue per calendar day are also calculated. The period has 128 weekdays and 52 weekend days.

In [7]:
period = pd.date_range(sales["order_date"].min(), sales["order_date"].max())
days_in_period = pd.Series(period.dayofweek >= 5).value_counts()      # True = weekend days

wk = sales.groupby("is_weekend").agg(
    orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    avg_order_revenue=("revenue", "mean"),
)
wk["days_in_period"] = [days_in_period[False], days_in_period[True]]
wk["orders_per_day"] = (wk["orders"] / wk["days_in_period"]).round(2)
wk["revenue_per_day"] = (wk["total_revenue"] / wk["days_in_period"]).round(0)
wk["revenue_share_%"] = (wk["total_revenue"] / wk["total_revenue"].sum() * 100).round(1)
wk.index = ["Weekday (Mon-Fri)", "Weekend (Sat-Sun)"]
wk["avg_order_revenue"] = wk["avg_order_revenue"].round(0)
wk

,orders,total_revenue,avg_order_revenue,days_in_period,orders_per_day,revenue_per_day,revenue_share_%
Weekday (Mon-Fri),723,51856200,71724.0,128,5.65,405127.0,72.2
Weekend (Sat-Sun),277,19936300,71972.0,52,5.33,383390.0,27.8


**Answer:**

| | Orders | Total revenue | Average order revenue |
|---|---|---|---|
| Weekday (Mon-Fri) | 723 | KES 51,856,200 | KES 71,724 |
| Weekend (Sat-Sun) | 277 | KES 19,936,300 | KES 71,972 |

Weekdays bring in more orders and more revenue in total (72.2% of revenue), but that is only because there are 128 weekdays against 52 weekend days. **Per day, weekends are slightly weaker:** 5.33 orders and KES 383,390 a day, against 5.65 orders and KES 405,127 on a weekday (5.4% less revenue). The average order is almost identical (KES 71,972 against KES 71,724, a 0.3% difference). Duka has **no weekend sales peak** in this data, so the data does not justify weekend-specific staffing or promotions.

## Task 6: Best and worst month

**Approach:** Revenue is totalled by month. The percentage difference is `(best - worst) / worst x 100`. Because June has 29 days of data and February has 28, revenue per day is also checked so the answer is not driven by month length.

In [8]:
monthly = sales.groupby("month").agg(total_revenue=("revenue", "sum"), orders=("order_id", "count")).reindex(month_order)
monthly["avg_order_value"] = (monthly["total_revenue"] / monthly["orders"]).round(0)
monthly["days_of_data"] = [31, 28, 31, 30, 31, 29]
monthly["revenue_per_day"] = (monthly["total_revenue"] / monthly["days_of_data"]).round(0)

best, worst = monthly["total_revenue"].idxmax(), monthly["total_revenue"].idxmin()
best_rev, worst_rev = monthly.loc[best, "total_revenue"], monthly.loc[worst, "total_revenue"]

print(f"Best month : {best} (KES {best_rev:,})")
print(f"Worst month: {worst} (KES {worst_rev:,})")
print(f"{best} is {(best_rev - worst_rev) / worst_rev * 100:.1f}% higher than {worst}")
print(f"{worst} is {(best_rev - worst_rev) / best_rev * 100:.1f}% lower than {best}")
monthly

Best month : April (KES 13,518,300)
Worst month: June (KES 10,903,400)
April is 24.0% higher than June
June is 19.3% lower than April


,total_revenue,orders,avg_order_value,days_of_data,revenue_per_day
month,,,,,
January,11786600,173,68131.0,31,380213.0
February,11772000,159,74038.0,28,420429.0
March,12396800,186,66649.0,31,399897.0
April,13518300,167,80948.0,30,450610.0
May,11415400,153,74610.0,31,368239.0
June,10903400,162,67305.0,29,375979.0


**Answer:** **April is the best month** (KES 13,518,300) and **June is the worst** (KES 10,903,400). April's revenue is **24.0% higher** than June's, or put the other way, June is 19.3% lower than April.

**Context:**

- **April's peak comes from bigger orders, not more of them.** April had only 167 orders (March had the most, 186), but its average order of KES 80,948 was the highest of any month.
- **June is partly a short month.** The data ends on 29 June, so June has one day fewer than April. Per day, April is still the best month (KES 450,610 a day), but the lowest is **May** (KES 368,239 a day), with June close behind (KES 375,979). The slowdown is therefore real and starts in May.
- **Overall pattern:** after April's peak, revenue fell 15.6% in May, to KES 11,415,400.

## Task 7: Premium online orders from Nairobi and Mombasa

**Approach:** Three conditions are combined with `&`, each in parentheses: `price_band` is Premium, `channel` is Online, and `branch` is in Nairobi or Mombasa (using `isin`). The result is exported to `outputs/premium_online_coast_capital.csv` with `index=False`.

In [9]:
mask = (
    (sales["price_band"] == "Premium")
    & (sales["channel"] == "Online")
    & (sales["branch"].isin(["Nairobi", "Mombasa"]))
)
premium_online = sales[mask].copy()

os.makedirs("outputs", exist_ok=True)
premium_online.to_csv("outputs/premium_online_coast_capital.csv", index=False)

online_revenue = sales.loc[sales["channel"] == "Online", "revenue"].sum()
print("Average order revenue by channel (all orders):")
print(sales.groupby("channel")["revenue"].agg(orders="count", avg_order_revenue="mean").round(0), "\n")
print("Premium online orders, Nairobi and Mombasa")
print("Orders:", len(premium_online))
print(f"Total revenue: KES {premium_online['revenue'].sum():,}")
print(f"Share of all revenue   : {premium_online['revenue'].sum() / sales['revenue'].sum():.1%}")
print(f"Share of online revenue: {premium_online['revenue'].sum() / online_revenue:.1%}")
print()
display(premium_online.groupby("branch").agg(orders=("order_id", "count"), revenue=("revenue", "sum")))
display(premium_online.groupby("product").agg(orders=("order_id", "count"), revenue=("revenue", "sum")))

check = pd.read_csv("outputs/premium_online_coast_capital.csv")
print("Exported file:", check.shape, "| first column:", check.columns[0])

Average order revenue by channel (all orders):
             orders  avg_order_revenue
channel                               
In-store        579            70759.0
Online          320            71207.0
Phone order     101            79576.0 

Premium online orders, Nairobi and Mombasa
Orders: 83
Total revenue: KES 10,218,000
Share of all revenue   : 14.2%
Share of online revenue: 44.8%



,orders,revenue
branch,,
Mombasa,33,3696000
Nairobi,50,6522000


,orders,revenue
product,,
Laptop,33,5850000
Tablet,50,4368000


Exported file: (83, 12) | first column: order_id


**Answer:** There are **83 Premium online orders** from Nairobi and Mombasa, worth **KES 10,218,000** in total. That is 14.2% of all Duka revenue and 44.8% of all online revenue, from 8.3% of all orders.

- **By branch:** Nairobi has 50 orders (KES 6,522,000) and Mombasa 33 (KES 3,696,000).
- **By product:** 50 are Tablets and 33 are Laptops.

This is the strongest argument for investing in online sales: nearly half of online revenue comes from high-priced products sold in the two biggest branches. The export has 83 rows and no index column.

## Task 8: Spreadsheet task and reconciliation with pandas

**Approach:** The workbook `excel/duka_branch_summary.xlsx` holds the same analysis in Excel:

- **Sales sheet:** the 1,000 orders as an Excel Table, with `revenue` as `=E2*G2`, a `region` column added with `XLOOKUP` from the **Branches** sheet (Nairobi is Central, Mombasa is Coast, Kisumu is Western, Nakuru is Rift Valley and Eldoret is North Rift), and conditional formatting that highlights orders above KES 100,000.
- **Summary sheet:** the branch revenue ranking using `SUMIFS`, a sorted bar chart with an insight title, and a branch by channel cross-tab, also built with `SUMIFS`.
- **PivotTable:** revenue (sum) with branch as rows and channel as columns, which must show the same grand totals as the cross-tab below.

The cell below reads the values stored in the workbook and compares them with the pandas results.

In [10]:
xl_file = "excel/duka_branch_summary.xlsx"
region_of = {"Nairobi": "Central", "Mombasa": "Coast", "Kisumu": "Western",
             "Nakuru": "Rift Valley", "Eldoret": "North Rift"}

# SUMIFS branch table from the workbook
xl_branch = pd.read_excel(xl_file, sheet_name="Summary", header=2, usecols="A:F", nrows=5).set_index("Branch")

pd_branch = sales.groupby("branch").agg(revenue=("revenue", "sum"), orders=("order_id", "count"))
pd_branch = pd_branch.loc[xl_branch.index]

print("Branch revenue (SUMIFS) matches pandas:", (xl_branch["Revenue (KES)"].values == pd_branch["revenue"].values).all())
print("Branch order counts match pandas      :", (xl_branch["Orders"].values == pd_branch["orders"].values).all())
print("XLOOKUP regions are as expected       :", (xl_branch["Region"].values == xl_branch.index.to_series().map(region_of).values).all())
print("Excel grand total:", f"KES {xl_branch['Revenue (KES)'].sum():,}", "| pandas:", f"KES {sales['revenue'].sum():,}")

# Branch x channel totals: the PivotTable must show these same numbers
xl_xtab = pd.read_excel(xl_file, sheet_name="Summary", header=12, usecols="A:E", nrows=6, index_col=0)
pd_xtab = pd.crosstab(sales["branch"], sales["channel"], values=sales["revenue"], aggfunc="sum",
                      margins=True, margins_name="Total")
pd_xtab = pd_xtab.reindex(index=xl_xtab.index, columns=xl_xtab.columns)

print("Branch x channel totals match pandas  :", (xl_xtab.values == pd_xtab.values).all())
print("Orders above KES 100,000 (highlighted in Excel):", (sales["revenue"] > 100_000).sum())
print("\nExpected PivotTable (revenue by branch and channel, KES):")
pd_xtab

Branch revenue (SUMIFS) matches pandas: True
Branch order counts match pandas      : True
XLOOKUP regions are as expected       : True
Excel grand total: KES 71,792,500 | pandas: KES 71,792,500


Branch x channel totals match pandas  : True
Orders above KES 100,000 (highlighted in Excel): 264

Expected PivotTable (revenue by branch and channel, KES):


,In-store,Online,Phone order,Total
Branch,,,,
Nairobi,14808400,7527400,3770300,26106100
Mombasa,8039600,4086800,1877300,14003700
Nakuru,6767300,4350400,852700,11970400
Kisumu,6468600,3370800,1028700,10868100
Eldoret,4885300,3450700,508200,8844200
Total,40969200,22786100,8037200,71792500


**Answer:** The Excel and pandas figures **match exactly**. Branch revenue and order counts from `SUMIFS` and `COUNTIFS` equal the pandas totals, the `XLOOKUP` regions are correct, and the grand total is KES 71,792,500 in both. The branch by channel table above is what the PivotTable shows: for example, Nairobi's In-store revenue is KES 14,808,400 and the Online column totals KES 22,786,100. The workbook was also recalculated independently in LibreOffice, and every `SUMIFS`, `COUNTIFS` and row-level revenue figure matched pandas.